# Pré-Processamento COCO 2017 

## Célula 0: Imports Globais

Rode esta célula primeiro para carregar todas as bibliotecas necessárias


In [1]:
# Verificar o status da GPU (se aplicável)
# !nvidia-smi

In [2]:
import json
import shutil
from pathlib import Path
import time

print("✅ Todas as bibliotecas carregadas com sucesso!")

✅ Todas as bibliotecas carregadas com sucesso!


## Célula 1: Configurações Iniciais

Rode esta célula para configurar os caminhos e as classes alvo

In [3]:
# ==========================================
# 1. CONFIGURAÇÕES DO PROJETO (JUPYTER)
# ==========================================
# Pega o diretório atual onde este notebook está rodando
BASE_DIR = Path.cwd() 
COCO_DIR = BASE_DIR / "coco_2017"
ANNOTATIONS_FILE = COCO_DIR / "annotations" / "instances_train2017.json"
IMAGES_DIR = COCO_DIR / "train2017"

OUTPUT_DIR = BASE_DIR / "coco_dataset"
OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_LABELS = OUTPUT_DIR / "labels"

# Nossa meta perfeita de balanceamento
IMAGENS_POR_CLASSE = 3000

# Dicionário do COCO original
COCO_CLASSES = {
    1: 'person',
    2: 'bicycle',
    3: 'car',
    4: 'motorcycle',
    5: 'bus'
}

# Nosso mapeamento para o YOLO (Obrigatoriamente começando em 0)
YOLO_MAPPING = {
    1: 0, # [0] person 
    2: 1, # [1] bicycle
    3: 2, # [2] car
    4: 3, # [3] motorcycle
    5: 4  # [4] bus
}

print(f"✅ Diretório base configurado para: {BASE_DIR}")
print(f"🎯 Meta: {IMAGENS_POR_CLASSE} imagens por classe alvo.")

✅ Diretório base configurado para: /home/kevin/Documents/TCC/COCO-2017
🎯 Meta: 3000 imagens por classe alvo.


## Célula 2: Funções de Suporte (Limpeza e Matemática)

Rode esta célula para registrar as funções na memória do Jupyter

In [4]:
# ==========================================
# 2. FUNÇÕES DE SUPORTE
# ==========================================
def setup_directories():
    """Limpa execuções anteriores e cria a estrutura do YOLO."""
    if OUTPUT_DIR.exists():
        print("🧹 Limpando diretório antigo...")
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
    OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)
    print(f"📁 Diretórios criados em: {OUTPUT_DIR}")

def convert_coco_to_yolo(bbox, img_width, img_height):
    """Converte [x_min, y_min, w, h] (COCO) para [x_center, y_center, norm_w, norm_h] (YOLO)."""
    x_min, y_min, w, h = bbox
    
    # Programação Defensiva: Garante que a caixa não vaze para fora da foto
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    w = min(w, img_width - x_min)
    h = min(h, img_height - y_min)

    x_center = (x_min + w / 2.0) / img_width
    y_center = (y_min + h / 2.0) / img_height
    norm_w = w / img_width
    norm_h = h / img_height

    return x_center, y_center, norm_w, norm_h

print("✅ Funções auxiliares carregadas!")

✅ Funções auxiliares carregadas!


## Célula 3: O Algoritmo de Mineração (O Cérebro do Script)

Rode esta célula. É aqui que o JSON é lido e a matemática de balanceamento e sobreposição de imagens acontece na RAM. Nenhuma foto é copiada ainda.

In [5]:
# ==========================================
# 3. LEITURA E BALANCEAMENTO LÓGICO
# ==========================================
start_time = time.time()
print("⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)")

with open(ANNOTATIONS_FILE, 'r') as f:
    coco_data = json.load(f)
print("✅ JSON carregado na memória!")

# Mapeamento rápido de imagens
images_info = {img['id']: img for img in coco_data['images']}

# Agrupa anotações por imagem (filtrando multidões borradas e focando nas 4 classes)
annotations_by_image = {}
for ann in coco_data['annotations']:
    cat_id = ann['category_id']
    if cat_id in COCO_CLASSES and not ann.get('iscrowd', 0):
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

# Dicionários de rastreamento para o nosso relatório final
selected_image_ids = set()
images_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}
objects_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}

print(f"🎯 Iniciando a busca exata de {IMAGENS_POR_CLASSE} imagens por classe...")

# Abordagem de preenchimento (Greedy)
for target_cat_id, class_name in COCO_CLASSES.items():
    for img_id, anns in annotations_by_image.items():
        if images_per_class[target_cat_id] >= IMAGENS_POR_CLASSE:
            break # Bateu a meta dessa classe, vai pra próxima
            
        has_target_class = any(a['category_id'] == target_cat_id for a in anns)
        
        if has_target_class and img_id not in selected_image_ids:
            selected_image_ids.add(img_id)
            
            # Conta os objetos físicos dentro dessa foto e atualiza o rastreador de imagens
            classes_in_this_image = set()
            for a in anns:
                c_id = a['category_id']
                if c_id in COCO_CLASSES:
                    objects_per_class[c_id] += 1
                    classes_in_this_image.add(c_id)
                    
            for c_id in classes_in_this_image:
                images_per_class[c_id] += 1

print(f"\n🔥 Mineração concluída em {time.time() - start_time:.2f} segundos!")
print(f"Total de imagens únicas selecionadas para extração: {len(selected_image_ids)}")

⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)


✅ JSON carregado na memória!
🎯 Iniciando a busca exata de 3000 imagens por classe...

🔥 Mineração concluída em 52.09 segundos!
Total de imagens únicas selecionadas para extração: 11240


## Célula 4: A Extração Física (Cópia das Imagens e Geração dos TXTs)

Rode esta célula. É ela que vai entrar na pasta gigante de 18 GB, copiar os milhares de arquivos selecionados e gerar as conversões do YOLO. Essa é a que mais demora.

In [6]:
# ==========================================
# 4. EXTRAÇÃO DE ARQUIVOS E ESCRITA DAS LABELS
# ==========================================
print("⚙️  Preparando diretórios de saída...")
setup_directories()

print("🚀 Iniciando a cópia física das imagens e geração dos .txt (Aguarde...)")
imgs_copiadas = 0

for img_id in selected_image_ids:
    img_data = images_info[img_id]
    file_name = img_data['file_name']
    img_width = img_data['width']
    img_height = img_data['height']
    
    src_img_path = IMAGES_DIR / file_name
    dst_img_path = OUTPUT_IMAGES / file_name
    
    if src_img_path.exists():
        shutil.copy2(src_img_path, dst_img_path)
        imgs_copiadas += 1
    else:
        continue # Ignora graciosamente se a foto não estiver no disco
        
    txt_name = Path(file_name).stem + ".txt"
    txt_path = OUTPUT_LABELS / txt_name
    
    with open(txt_path, 'w') as txt_file:
        for ann in annotations_by_image[img_id]:
            cat_id = ann['category_id']
            # Filtro defensivo: Só escreve no txt se for uma das nossas 5 classes alvo
            if cat_id in YOLO_MAPPING:
                yolo_class = YOLO_MAPPING[cat_id]
                x_c, y_c, w, h = convert_coco_to_yolo(ann['bbox'], img_width, img_height)
                txt_file.write(f"{yolo_class} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

print(f"✅ Fim da extração física! {imgs_copiadas} imagens copiadas com sucesso.")

⚙️  Preparando diretórios de saída...
🧹 Limpando diretório antigo...


📁 Diretórios criados em: /home/kevin/Documents/TCC/COCO-2017/coco_dataset
🚀 Iniciando a cópia física das imagens e geração dos .txt (Aguarde...)
✅ Fim da extração física! 11240 imagens copiadas com sucesso.


## Célula 5: A Geração do Relatório Tabular do TCC

Rode esta célula. Ela vai imprimir a tabela com relatório final

In [7]:
# ==========================================
# 5. RELATÓRIO FINAL
# ==========================================
print("\n" + "="*95)
print("📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (COCO BALANCEADO - TREINAMENTO DO ZERO)")
print("="*95)
print(f"{'Classe ID':<10} {'Nome da Classe':<15} {'Total de Objetos':<20} {'Total de Imagens':<20} {'Média de Objetos/Imagem'}")
print("-" * 95)

total_objetos_geral = 0
for cat_id, name in COCO_CLASSES.items():
    yolo_id = YOLO_MAPPING[cat_id]
    total_obj = objects_per_class[cat_id]
    total_img = images_per_class[cat_id]
    media = total_obj / total_img if total_img > 0 else 0
    total_objetos_geral += total_obj
    
    print(f"{yolo_id:<10} {name:<15} {total_obj:<20} {total_img:<20} {media:.2f}")

print("="*95)
print("\n📈 RESUMO GERAL")
print("="*95)
print(f"✓ Total de imagens processadas fisicamente: {imgs_copiadas}")
print(f"✓ Total de objetos detectados (Bounding Boxes): {total_objetos_geral}")

media_geral = total_objetos_geral / imgs_copiadas if imgs_copiadas > 0 else 0
print(f"✓ Média de objetos por imagem no Dataset: {media_geral:.2f}")
print(f"✓ Número de classes ativas: {len(COCO_CLASSES)}")
print("="*95)
print(f"\n🎉 Tudo pronto! Pode zipar a pasta '{OUTPUT_DIR.name}' e fazer upload no Drive!")


📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (COCO BALANCEADO - TREINAMENTO DO ZERO)
Classe ID  Nome da Classe  Total de Objetos     Total de Imagens     Média de Objetos/Imagem
-----------------------------------------------------------------------------------------------
0          person          37427                7588                 4.93
1          bicycle         6605                 3001                 2.20
2          car             14524                3996                 3.63
3          motorcycle      7146                 3006                 2.38
4          bus             5129                 2986                 1.72

📈 RESUMO GERAL
✓ Total de imagens processadas fisicamente: 11240
✓ Total de objetos detectados (Bounding Boxes): 70831
✓ Média de objetos por imagem no Dataset: 6.30
✓ Número de classes ativas: 5

🎉 Tudo pronto! Pode zipar a pasta 'coco_dataset' e fazer upload no Drive!


## Célula 6: Compactação do Dataset (Preparação para Upload)

Rode esta célula para criar um arquivo ZIP do dataset processado, pronto para fazer upload no Google Drive ou em qualquer outro repositório


In [ ]:
# ==========================================
# CONFIGURAÇÃO DA COMPACTAÇÃO
# ==========================================
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "coco_dataset"  
ZIP_OUTPUT = BASE_DIR / "coco_dataset"  

if DATASET_DIR.exists():
    print(f"📦 Iniciando a compactação da pasta '{DATASET_DIR.name}'...")
    start_time = time.time()
    
    # Compacta a pasta de forma eficiente
    shutil.make_archive(ZIP_OUTPUT, 'zip', DATASET_DIR)
    
    print("\n" + "="*60)
    print(f"✅ SUCESSO! Arquivo '{ZIP_OUTPUT.name}.zip' gerado com sucesso.")
    print(f"📍 Localização: {BASE_DIR / (ZIP_OUTPUT.name + '.zip')}")
    print(f"⏱️  Tempo de execução: {time.time() - start_time:.2f} segundos.")
    print("="*60)
    print("\n🚀 PRÓXIMOS PASSO Machine Learning Operations (MLOps):")
    print("Abra o seu Google Drive e faça o upload desse arquivo 'coco_dataset.zip'.")
else:
    print(f"❌ Erro: O diretório '{DATASET_DIR}' não foi encontrado no caminho atual.")

📦 Iniciando a compactação da pasta 'coco_dataset'...

✅ SUCESSO! Arquivo 'coco_dataset.zip' gerado com sucesso.
📍 Localização: /home/kevin/Documents/TCC/COCO-2017/coco_dataset.zip
⏱️  Tempo de execução: 235.54 segundos.

🚀 PRÓXIMOS PASSO Machine Learning Operations (MLOps):
1. Abra o seu Google Drive e faça o upload desse arquivo 'coco_dataset.zip'.
